# Latih ulang detektor (CenterNet 3D bbox) dengan data kamera sendiri

Langkah sebelum notebook ini (di VM):
1. Beri label: `centernet3d_bbox/tools/label_detector.py` → `~/dataset_deteksi/<nama>`
2. Kemas: `python3 centernet3d_bbox/tools/pack_detector_training.py --data ~/dataset_deteksi/<nama>` → `~/paket_latih_detektor.zip`
3. Unggah zip ke Google Drive: **MyDrive/TA_detektor/paket_latih_detektor.zip**

Di Colab: **Runtime → Change runtime type → GPU (T4)**, lalu jalankan sel satu per satu.
Hasil (`centernet_lokal.pth` dan log) tersimpan di **MyDrive/TA_detektor/**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'TIDAK ADA GPU: ganti runtime ke GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/TA_detektor'
import os; print(os.listdir(DRIVE))

In [ ]:
!rm -rf /content/kerja && mkdir -p /content/kerja
!unzip -q "$DRIVE/paket_latih_detektor.zip" -d /content/kerja
!ls /content/kerja /content/kerja/data /content/kerja/bobot
!for d in /content/kerja/data/*; do echo "$d: $(ls $d/converted/images | wc -l) gambar"; done

## Latih
- `--epochs 10`, lr 5e-5 turun ×0.1 setiap 5 epoch. Bobot disimpan hanya bila **val loss data lokal** membaik.
- Data session0 ikut dicampur (bila ada di paket) supaya kemampuan lama tidak hilang; loss validasinya juga dicetak.
- Baris `sebelum lokal: loss=...` adalah titik awal: bobot lama pada data lokal. Yang dicari: loss lokal turun jauh,
  loss session0 tidak naik banyak.

In [ ]:
import glob, os
data = ' '.join('--data ' + d for d in sorted(glob.glob('/content/kerja/data/*')))
s0 = '--session0 /content/kerja/session0' if os.path.isdir('/content/kerja/session0') else ''
init = glob.glob('/content/kerja/bobot/*.pth')[0]
OUT = DRIVE + '/centernet_lokal.pth'
cmd = f'cd /content/kerja/kode && python train_local.py {data} {s0} --init {init} --out {OUT} --epochs 10 --batch-size 8 --lr 5e-5 --logdir {DRIVE}'
print(cmd)
!{cmd}

In [ ]:
import pandas as pd, glob, matplotlib.pyplot as plt
log = sorted(glob.glob(DRIVE + '/latih_lokal_*.csv'))[-1]
df = pd.read_csv(log); print(df.to_string(index=False))
ax = df.plot(x='epoch', y=[c for c in ['latih_loss', 'lokal_loss', 's0_loss'] if df[c].notna().any()], marker='o', figsize=(7, 4))
ax.set_ylabel('loss'); ax.grid(alpha=.3); plt.title('Loss latih detektor'); plt.show()

## Setelah selesai
1. Unduh `MyDrive/TA_detektor/centernet_lokal.pth` dan log `latih_lokal_*.txt`.
2. Di VM, taruh di `~/3d-vehicle-detection-counting/data/weights/detector/` (nama tetap `centernet_lokal.pth`).
3. Di web, halaman **Proses**: pilih model **centernet_lokal.pth**. Deteksi akan dihitung ulang (belum ada cache, lambat di CPU).
4. Bandingkan hitungan dengan model lama memakai `tools/compare_manual_count.py`.